### Using the HadISD datset (version 3.4.0.2023f) with PyEarthTools
HadISD is a global sub-daily dataset based on the ISD dataset from NOAA's NCEI. As well as station selection criteria, a suite of quality control tests has been run on the major climatological variables.

The dataset can be downloaded here: https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

In [ ]:
import datetime
from pathlib import Path

import pyearthtools.pipeline as petpipe
import pyearthtools.data as petdata
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

In [ ]:
# train/validation/test split dates
train_start = "1970-01-01T00"
train_end = "2022-12-31T23"

In [ ]:
varname_val_map = {
        "total_cloud_cover": -999., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.
    }

# Probably sensible to accetp a list since some variables might have multiple values
# This would look like:
# varname_val_map = {
#         "total_cloud_cover": [-888., -999.],
#         "low_cloud_cover": [-888., -999.],
#         "mid_cloud_cover": [-999.],
#         "high_cloud_cover": [-999.]
#     }

In [ ]:
flagged_labels = [
        'temperatures', 'dewpoints', 'slp',
        'stnlp', 'windspeeds', 'winddirs', 
        'total_cloud_cover', 'low_cloud_cover', 'mid_cloud_cover', 
        'high_cloud_cover', 'precip1_depth', 'precip2_depth', 
        'precip3_depth', 'precip6_depth', 'precip9_depth',
        'precip12_depth', 'precip15_depth', 'precip18_depth', 
        'precip24_depth'
    ]

date_range=(datetime.datetime(1986,11,20,12,0), datetime.datetime(1986,11,21,0,0))

In [ ]:
hadisd = HadISDIndex()
all_stations = hadisd.get_all_station_ids(Path("/Users/joelmiller/Projects/data/hadisd"))
all_stations_ordered = sorted(all_stations)
print(f"Total number of stations: {len(all_stations_ordered)}")

In [ ]:
first_ten = all_stations_ordered[:10]
first_ten

In [ ]:
data_prep_pipe = petpipe.Pipeline(
    # petdata.archive.hadisd(("010010-99999"), variables = ["total_cloud_cover", "temperatures", "flagged_obs"]),
    # petdata.archive.hadisd(station = ["010014-99999"], variables = ["total_cloud_cover", "temperatures", "flagged_obs"]),
    
    # Current options for indexing
    #petdata.archive.hadisd(["010014-99999", "010010-99999", "010030-99999"]),    
    petdata.archive.hadisd("010010-99999", variables=["total_cloud_cover", "temperatures", "flagged_obs"]),

    petdata.transforms.values.AddFlaggedObs(flagged_labels),
    petdata.transforms.values.SetMissingToNaN(varname_val_map),
    
    # Possible ways of indexing in the future:
    # petdata.archive.hadisd(country_code = "24", station_id = "010010", nearest_neighbors = 12, lon= 0.0, lat = 0.0), # Geospatial fence 
    # petdata.archive.hadisd("010014-99999"), # Geospatial fence
)
data_prep_pipe

# Don't know what combination of stations you want when selecting nearest neighbours, so want to cahce on a per-station basis


In [ ]:
ds = data_prep_pipe["1970-01-01T06"]
ds

In some cases you will get the following error when passing a date time to a pipeline object: `IndexWarning: Could not find time in dataset to select on. Petdt('1931-01-01T07')`<br>

This indicates that data for the datetime you chose does not exist. In this case PET will load all data from your station selection.

In [ ]:
# show total_cloud_cover from ds
tcc = ds["total_cloud_cover"]
tcc

# For specific Stations

In [ ]:
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex
from pathlib import Path
hadisd = HadISDIndex(["010014-99999", "010010-99999", "010030-99999"])
paths = hadisd.filesystem()
paths


In [ ]:
file_paths = list(paths.values())
file_paths

In [ ]:
ds = hadisd.load(file_paths)
ds

# For all stations

In [ ]:
hadisd = HadISDIndex("all")

all_station_ids = hadisd.get_all_station_ids(Path("/Users/joelmiller/Projects/data/hadisd"))
print(all_station_ids[:10])
print(len(all_station_ids))


In [ ]:
hadisd = HadISDIndex(all_station_ids)
paths = hadisd.filesystem() 
paths = list(paths.values())

In [ ]:
ds_all = hadisd.load(paths)
ds_all

In [ ]:
import pyearthtools.data as petdata
# select only total_cloud_cover
ds_all_tcc = ds_all.pipe(petdata.transforms.variables.Select("total_cloud_cover"))
ds_all_tcc


In [ ]:
# Manuall apply the transform
varname_val_map = {
        "total_cloud_cover": -999., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.
    }

ds_all_tcc_nan = ds_all.pipe(petdata.transforms.values.SetMissingToNaN(varname_val_map))
ds_all_tcc_nan

In [ ]:
tcc = ds_all["total_cloud_cover"]
tcc

# Use the section below to build tests to check that all pipeline steps are working as expected

## Test SetMissingToNaN

In [ ]:
import xarray as xr
from pyearthtools.data.transforms.values import SetMissingToNaN
import numpy as np

def test_set_missing_to_nan():
    data = xr.Dataset({
        "total_cloud_cover": ("time", [0, -999, 50]),
        "low_cloud_cover": ("time", [10, -999, 20]),
    })

    varname_val_map = {
        "total_cloud_cover": -99.0,
        "low_cloud_cover": -999.0,
    }

    transform = SetMissingToNaN(varname_val_map)
    transformed_data = transform.apply(data)

    if np.isnan(transformed_data["total_cloud_cover"].data[1]):
        print("Total cloud cover at index 1 is NaN")
    else:
        print("Total cloud cover at index 1 is not NaN")
    
    if np.isnan(transformed_data["low_cloud_cover"].data[1]):
        print("Low cloud cover at index 1 is NaN")
    
    if transformed_data["total_cloud_cover"].data[2] == 50:
        print("Total cloud cover at index 2 is 50")

In [ ]:
test_set_missing_to_nan()

## Test AddFlaggedObs

In [ ]:
import xarray as xr
import numpy as np
from pyearthtools.data.transforms.values import AddFlaggedObs

def test_add_flagged_obs():
    # Mock dataset with flagged observations
    data = xr.Dataset(
        {
            "temperatures": (("time",), [np.nan, 15.0, np.nan]),
            "dewpoints": (("time",), [np.nan, 10.0, np.nan]),
            "flagged_obs": (
                ("time", "flagged"),
                [
                    [20.0, np.nan],  # Flagged data for time=0
                    [np.nan, np.nan],  # No flagged data for time=1
                    [25.0, 12.0],  # Flagged data for time=2
                ],
            ),
        },
        coords={
            "time": [0, 1, 2],
            "flagged": [0, 1],  # Indices for flagged variables
        },
    )

    # Add flagged_value attribute to variables
    data["temperatures"].attrs["flagged_value"] = -999.0
    data["dewpoints"].attrs["flagged_value"] = -999.0

    # Define flagged labels corresponding to the flagged dimension
    flagged_labels = ["temperatures", "dewpoints"]

    # Apply the AddFlaggedObs transform
    transform = AddFlaggedObs(flagged_labels)
    transformed_data = transform.apply(data)

    # Assert that flagged data has been restored
    assert np.allclose(
        transformed_data["temperatures"].data, [20.0, 15.0, 25.0], equal_nan=True
    )
    assert np.allclose(
        transformed_data["dewpoints"].data, [np.nan, 10.0, 12.0], equal_nan=True
    )

    print("Test passed: Flagged observations were correctly restored.")

# Run the test
test_add_flagged_obs()